<a href="https://colab.research.google.com/github/VuTuanAnh0949/RAG-LangChain-Ablation-Study/blob/main/Test_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cài đặt thư viện


In [ ]:
!pip install -q ragas>=0.2.0 langchain langchain-community langchain-openai
!pip install -q openai pypdf rapidfuzz pandas pymupdf4llm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 26.9 MB/s eta 0:00:00


## Nhập thư viện cần thiết

In [ ]:
import os
import re
import json
import warnings
import pymupdf4llm
import pandas as pd
from pathlib import Path
from google.colab import userdata
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.testset import TestsetGenerator
from langchain_openai import OpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.document_loaders import DirectoryLoader, TextLoader

warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Cấu hình API Key

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


## Tải bộ dữ liệu

In [ ]:
!gdown 19C6FVRVTn4bpe7HFoutClQO-1VXn2L7w
!unzip -q data.zip


Downloading...
From (original): https://drive.google.com/uc?id=19C6FVRVTn4bpe7HFoutClQO-1VXn2L7w
From (redirected): https://drive.google.com/uc?id=19C6FVRVTn4bpe7HFoutClQO-1VXn2L7w&confirm=t&uuid=b2bf2eb2-e8f2-41eb-9170-5802352ec33e
To: /content/data.zip
100% 133M/133M [00:03<00:00, 41.4MB/s]


## Chuyển đổi PDF sang Markdown

In [ ]:
def convert_pdfs_to_markdown(pdf_dir, output_dir=None):
    pdf_path = Path(pdf_dir)
    output_path = Path(output_dir) if output_dir else pdf_path / "markdown"
    output_path.mkdir(exist_ok=True)
    pdf_files = list(pdf_path.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")
    converted = 0
    for pdf_file in pdf_files:
        try:
            print(f"Converting: {pdf_file.name}")
            md_text = pymupdf4llm.to_markdown(str(pdf_file))
            md_file = output_path / f"{pdf_file.stem}.md"
            with open(md_file, 'w', encoding='utf-8') as f:
                f.write(md_text)
            converted += 1
        except Exception as e:
            print(f"Error: {e}")
    print(f"\nConverted {converted}/{len(pdf_files)} files")
    print(f"Output: {output_path}")
    return str(output_path)

PDF_DIR = "./AIO_Docs"
md_dir = convert_pdfs_to_markdown(PDF_DIR)


Found 45 PDF files
Converting: Interpolation_Missing_Data.pdf
=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=9/10.

Converting: AI_Vietnamese_Poetry.pdf
=== Document parser messages ===
                                                             Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=3/4.
OCR on page.number=6/7.
OCR on page.number=7/8.
OCR on page.number=8/9.
OCR on page.number=11/12.
OCR on page.number=14/15.
OCR on page.number=15/16.
OCR on page.number=16/17.
OCR on page.number=18/19.

Converting: Applied_Statistics_AI.pdf
=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                           Using T

## Đọc dữ liệu markdown

In [ ]:
DATA_DIR = "./AIO_Docs/markdown"
loader = DirectoryLoader(
    DATA_DIR,
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)
docs = loader.load()
print(f"\nLoaded {len(docs)} Markdown documents")


100%|██████████| 45/45 [00:00<00:00, 1914.02it/s]


Loaded 45 Markdown documents


## Thiết lập các mô hình

In [ ]:
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", max_completion_tokens=8192))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


## Tạo bộ dữ liệu đánh giá

In [ ]:
TESTSET_SIZE = 50

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings
)
dataset = generator.generate_with_langchain_docs(
    docs,
    testset_size=TESTSET_SIZE
)

print(f"Generating {TESTSET_SIZE} samples from {len(docs)} documents...")
print(f"\nGenerated {len(dataset)} samples!")

Applying HeadlinesExtractor:   0%|          | 0/45 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/45 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/45 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/955 [00:00<?, ?it/s]

ERROR:ragas.async_utils:Task failed with RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-8wnTmgNU4OHbyDoFmlhyDF42 on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
ERROR:ragas.async_utils:Task failed with RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-8wnTmgNU4OHbyDoFmlhyDF42 on tokens per min (TPM): Limit 200000, Used 198828, Requested 1199. Please try again in 8ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-8wnTmgNU4OHbyDoFmlhyDF42 on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}

## Lưu thành file json

In [ ]:
records = df.to_dict(orient="records")
with open("ragas_testset.json", 'w', encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2, default=str)
